In [1]:
import pandas as pd
import plotly.express as px
from gerar_mapa import BASE
import numpy as np
import plotly.io as pio


# ============================================================
# CONFIGURAÇÕES
# ============================================================

EMPRESAS = ["cbo", "bram", "starnav"]

# Cor padrão do tema OFI
COR_FUNDO = "#0B1F3A"
COR_TEXTO = "#FFFFFF"


# ============================================================
# PROCESSAMENTO DAS EMPRESAS
# ============================================================

for EMPRESA in EMPRESAS:

    ARQUIVO = (
        BASE
        / "dados"
        / f"SURVEY_{EMPRESA}_D2.xlsx"
    )


    # ========================================================
    # DIRETÓRIO DE SAÍDA
    # ========================================================

    PASTA_SAIDA = (
        BASE
        / "graficos"
        / "gantt_atendimento"
        / EMPRESA
    )

    PASTA_SAIDA.mkdir(
        parents=True,
        exist_ok=True
    )


    # ========================================================
    # LEITURA DOS DADOS
    # ========================================================

    abas = pd.read_excel(
        ARQUIVO,
        sheet_name=None
    )


    # ========================================================
    # PROCESSAMENTO DAS ABAS
    # ========================================================

    for nome_aba, df in abas.items():

        # ----------------------------------------------------
        # Ignora aba de posições
        # ----------------------------------------------------

        if nome_aba.lower() == "posicoes":
            continue


        print(f"\n{'=' * 70}")
        print(f"Processando Gantt: {nome_aba}")
        print(f"{'=' * 70}")


        # ====================================================
        # PREPARAÇÃO DOS DADOS
        # ====================================================

        colunas_necessarias = [
            "evidencia_proximidade",
            "data_reportada",
            "unidade_proxima"
        ]


        colunas_faltantes = [
            coluna
            for coluna in colunas_necessarias
            if coluna not in df.columns
        ]


        if colunas_faltantes:

            print(
                f"Colunas ausentes na aba {nome_aba}: "
                f"{colunas_faltantes}"
            )

            continue


        # ----------------------------------------------------
        # Converte evidência para número
        # ----------------------------------------------------

        df["evidencia_proximidade"] = pd.to_numeric(
            df["evidencia_proximidade"],
            errors="coerce"
        )


        # ----------------------------------------------------
        # Converte data
        # ----------------------------------------------------

        df["data_reportada"] = pd.to_datetime(
            df["data_reportada"],
            dayfirst=True,
            errors="coerce"
        )


        # ----------------------------------------------------
        # Remove linhas sem data
        # ----------------------------------------------------

        df_base = (
            df
            .dropna(subset=["data_reportada"])
            .sort_values("data_reportada")
            .reset_index(drop=True)
        )


        # ----------------------------------------------------
        # Evidência suficiente
        # ----------------------------------------------------

        df_base["atendimento"] = (
            df_base["evidencia_proximidade"] >= 0.5
        )


        # ====================================================
        # CONSTRUÇÃO DOS EVENTOS
        # ====================================================

        registros_gantt = []

        evento_atual = None


        for i in range(len(df_base)):

            linha = df_base.iloc[i]

            data = linha["data_reportada"]

            evidencia = linha["evidencia_proximidade"]

            unidade = linha["unidade_proxima"]


            # =================================================
            # EVIDÊNCIA INSUFICIENTE
            # =================================================

            if pd.isna(evidencia) or evidencia < 0.5:

                if evento_atual is not None:

                    evento_atual["Fim"] = data

                    if (
                        evento_atual["Fim"]
                        > evento_atual["Inicio"]
                    ):

                        registros_gantt.append(
                            evento_atual.copy()
                        )

                    evento_atual = None

                continue


            # =================================================
            # UNIDADE INEXISTENTE
            # =================================================

            if pd.isna(unidade):

                if evento_atual is not None:

                    evento_atual["Fim"] = data

                    if (
                        evento_atual["Fim"]
                        > evento_atual["Inicio"]
                    ):

                        registros_gantt.append(
                            evento_atual.copy()
                        )

                    evento_atual = None

                continue


            # =================================================
            # PRIMEIRO EVENTO
            # =================================================

            if evento_atual is None:

                evento_atual = {

                    "Inicio": data,

                    "Fim": data,

                    "Origem": unidade,

                    "Destino": unidade,

                    "Tipo": "Parado",

                    "Atividade": unidade

                }

                continue


            # =================================================
            # UNIDADE ATUAL
            # =================================================

            origem_atual = evento_atual["Destino"]


            # =================================================
            # MESMA UNIDADE
            # =================================================

            if unidade == origem_atual:

                evento_atual["Fim"] = data

                continue


            # =================================================
            # MUDANÇA DE UNIDADE
            # =================================================

            evento_atual["Fim"] = data


            if (
                evento_atual["Fim"]
                > evento_atual["Inicio"]
            ):

                registros_gantt.append(
                    evento_atual.copy()
                )


            # -------------------------------------------------
            # Novo evento de trânsito
            # -------------------------------------------------

            evento_atual = {

                "Inicio": data,

                "Fim": data,

                "Origem": origem_atual,

                "Destino": unidade,

                "Tipo": "Trânsito",

                "Atividade": (
                    f"{origem_atual} → {unidade}"
                )

            }


        # ====================================================
        # FECHA EVENTO FINAL
        # ====================================================

        if evento_atual is not None:

            ultima_data = (
                df_base.iloc[-1]["data_reportada"]
            )

            evento_atual["Fim"] = ultima_data


            if (
                evento_atual["Fim"]
                > evento_atual["Inicio"]
            ):

                registros_gantt.append(
                    evento_atual.copy()
                )


        # ====================================================
        # DATAFRAME DO GANTT
        # ====================================================

        df_gantt = pd.DataFrame(
            registros_gantt
        )


        if df_gantt.empty:

            print(
                f"Nenhum evento válido: {nome_aba}"
            )

            continue


        # ====================================================
        # REMOVE INTERVALOS INVÁLIDOS
        # ====================================================

        df_gantt = df_gantt[
            df_gantt["Fim"]
            > df_gantt["Inicio"]
        ].reset_index(drop=True)


        if df_gantt.empty:

            print(
                f"Nenhum intervalo válido: {nome_aba}"
            )

            continue


        # ====================================================
        # DURAÇÃO
        # ====================================================

        df_gantt["Duracao"] = (
            df_gantt["Fim"]
            - df_gantt["Inicio"]
        )


        # ====================================================
        # ID DO EVENTO
        # ====================================================

        df_gantt["id_evento"] = np.arange(
            len(df_gantt)
        )


        # ====================================================
        # EIXO Y
        # ====================================================

        df_gantt["Atividade_Y"] = (

            df_gantt["id_evento"]
            .astype(str)

            + " - "

            + df_gantt["Atividade"]
        )


        ordem_eventos = (
            df_gantt["Atividade_Y"]
            .tolist()
        )


        # ====================================================
        # CORES DAS ATIVIDADES
        # ====================================================

        cores_tipo = {

            "Trânsito": "#1CA3EC",

            "Parado": "#D4AF37"

        }


        # ====================================================
        # GANTT
        # ====================================================

        fig = px.timeline(

            df_gantt,

            x_start="Inicio",

            x_end="Fim",

            y="Atividade_Y",

            color="Tipo",

            color_discrete_map=cores_tipo,

            hover_data={

                "Origem": True,

                "Destino": True,

                "Duracao": True,

                "Tipo": True,

                "Inicio": True,

                "Fim": True,

                "Atividade_Y": False

            },

            category_orders={

                "Atividade_Y": ordem_eventos

            }

        )


        # ====================================================
        # EIXO X
        # ====================================================

        fig.update_xaxes(

            title_text="Data / Hora",

            title_font=dict(
                size=18,
                color=COR_TEXTO
            ),

            tickfont=dict(
                size=14,
                color=COR_TEXTO
            ),

            tickformat="%d/%m %H:%M",

            dtick=6 * 60 * 60 * 1000,

            automargin=True,

            color=COR_TEXTO,

            gridcolor="rgba(255,255,255,0.15)",

            zerolinecolor="rgba(255,255,255,0.20)"

        )


        # ====================================================
        # EIXO Y
        # ====================================================

        fig.update_yaxes(

            title_text="Atividade",

            title_font=dict(
                size=18,
                color=COR_TEXTO
            ),

            tickfont=dict(
                size=14,
                color=COR_TEXTO
            ),

            automargin=True,

            color=COR_TEXTO,

            gridcolor="rgba(255,255,255,0.08)",

            zerolinecolor="rgba(255,255,255,0.15)"

        )


        # ====================================================
        # ALTURA DO GANTT
        # ====================================================

        altura_gantt = max(
            500,
            len(df_gantt) * 35
        )


        # ====================================================
        # LAYOUT
        # ====================================================

        fig.update_layout(

            autosize=True,

            height=altura_gantt,

            # -----------------------------------------------
            # FUNDO
            # -----------------------------------------------

            paper_bgcolor=COR_FUNDO,

            plot_bgcolor=COR_FUNDO,

            # -----------------------------------------------
            # MARGENS
            # -----------------------------------------------

            margin=dict(

                l=220,

                r=30,

                t=100,

                b=60

            ),

            # -----------------------------------------------
            # FONTE PADRÃO
            # -----------------------------------------------

            font=dict(

                size=16,

                family="Arial",

                color=COR_TEXTO

            ),

            # -----------------------------------------------
            # TÍTULO
            # -----------------------------------------------

            title=dict(

                text=f"""
                <b>Atendimento Offshore</b><br>
                <span style='font-size:16px'>{nome_aba}</span>
                """,

                font=dict(

                    size=20,

                    color=COR_TEXTO

                ),

                x=0.5,

                xanchor="center"

            ),

            # -----------------------------------------------
            # LEGENDA
            # -----------------------------------------------

            legend=dict(

                title="Tipo de atendimento",

                title_font=dict(

                    size=14,

                    color=COR_TEXTO

                ),

                font=dict(

                    size=13,

                    color=COR_TEXTO

                ),

                bgcolor="rgba(0,0,0,0)"

            )

        )


        # ====================================================
        # BARRAS
        # ====================================================

        fig.update_traces(

            marker_line_color=COR_FUNDO,

            marker_line_width=1

        )


        # ====================================================
        # NOME DO ARQUIVO
        # ====================================================

        def normalizar_nome(nome_aba):
            return (
                str(nome_aba)
                .strip()
                .lower()
                .replace("/", "_")
                .replace("\\", "_")
                .replace(" ", "_")
            )

    
    
    # nome_arquivo = ( str(nome_aba)
          #  .strip()
          #  .lower()
          #  .replace("/", "_")
          #  .replace("\\", "_")
          #  .replace(" ", "_")
      #  )
        nome_arquivo = normalizar_nome(nome_aba)

        arquivo_saida = ( PASTA_SAIDA / f"{nome_arquivo}.html" )


        # ====================================================
        # SALVAMENTO
        # ====================================================

        fig.write_html(

            arquivo_saida,

            full_html=True,

            include_plotlyjs=True,

            config={

                "responsive": True,

                "displayModeBar": True,

                "scrollZoom": False

            }

        )


        print(
            f"Gantt salvo em:\n{arquivo_saida}"
        )
        


Processando Gantt: CBO ALESSANDRA
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\cbo\cbo_alessandra.html

Processando Gantt: CBO ALIANCA
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\cbo\cbo_alianca.html

Processando Gantt: CBO ANITA
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\cbo\cbo_anita.html

Processando Gantt: CBO ARPOADOR
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\cbo\cbo_arpoador.html

Processando Gantt: CBO CAMPOS
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\cbo\cbo_campos.html

Processando Gantt: CBO CAROLINA
Gantt salvo em:
C:\Users\

Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\bram\bram_bravo.html

Processando Gantt: BRAM BREEZE
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\bram\bram_breeze.html

Processando Gantt: BRAM BUCK
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\bram\bram_buck.html

Processando Gantt: BRAM BUZIOS
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\bram\bram_buzios.html

Processando Gantt: BRAM HERO
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\bram\bram_hero.html

Processando Gantt: BRAM POWER
Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e 

Gantt salvo em:
C:\Users\roger\OneDrive - cefet-rj.br\Empresas e entrevistas\projeto logistica offshore\ofi\graficos\gantt_atendimento\starnav\starnav_volans.html
